# Fine-Tune T5Gemma (Encoder-Decoder Gemma) for Arabic Summarization

# Table of Contents

<a name='1'></a>
## 1 - Set up Kernel, Load Required Dependencies, Dataset and LLM

<a name='1.1'></a>
### 1.1 - Set up Kernel and Required Dependencies

In [1]:
%pip install -q -U \
    transformers\
    datasets \
    peft \
    accelerate\
    evaluate\
    rouge_score\
    bitsandbytes\
    sentencepiece



Note: you may need to restart the kernel to use updated packages.


In [58]:
%pip install -q bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [59]:
import os
import gc
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel,prepare_model_for_kbit_training
import evaluate

import numpy as np


# Check GPU
assert torch.cuda.is_available(), "No GPU found! Enable T4: Runtime -> Change runtime type"
DEVICE = torch.device("cuda")
print(f"GPU  : {torch.cuda.get_device_name(0)}")


GPU  : Tesla T4


<a name='1.2'></a>
### 1.2 - Load Dataset and LLM


In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_API")

login(token=hf_token)
print("Logged in via Kaggle secrets.")

Logged in via Kaggle secrets.


In [4]:
#from huggingface_hub import login
#from google.colab import userdata

#login(token=userdata.get("HF_TOKEN"))
#print("Logged in via Colab secrets.")


In [5]:
MODEL_ID = "google/t5gemma-2-270m-270m"
DATASET_ID = "BounharAbdelaziz/Arabic-Synthetic-Summarization-Dataset-Filtered"

TEXT_COL    = "text"
SUMMARY_COL = "summary"

dataset = load_dataset(DATASET_ID)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'summary_model_name', 'tokenizer_name', 'dataset_source', 'sequence_length'],
        num_rows: 3526
    })
    validation: Dataset({
        features: ['text', 'summary', 'summary_model_name', 'tokenizer_name', 'dataset_source', 'sequence_length'],
        num_rows: 437
    })
    test: Dataset({
        features: ['text', 'summary', 'summary_model_name', 'tokenizer_name', 'dataset_source', 'sequence_length'],
        num_rows: 444
    })
})


In [6]:
from transformers import AutoProcessor


In [7]:
# Use AutoProcessor instead of AutoTokenizer
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
print(f"Tokenizer loaded | vocab size: {tokenizer.vocab_size:,}")

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Tokenizer loaded | vocab size: 262,144


In [31]:
print(f"Loading model: {MODEL_ID}")
#bnb_config = BitsAndBytesConfig(
 #   load_in_4bit              = True,
  #  bnb_4bit_use_double_quant = True,
  #  bnb_4bit_quant_type       = "nf4",
  #  bnb_4bit_compute_dtype    = torch.bfloat16,
#)


original_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
   # quantization_config = bnb_config,
    torch_dtype = torch.bfloat16,
    device_map= "cuda"
)
original_model

Loading model: google/t5gemma-2-270m-270m


Loading weights:   0%|          | 0/911 [00:00<?, ?it/s]

T5Gemma2ForConditionalGeneration(
  (model): T5Gemma2Model(
    (encoder): T5Gemma2Encoder(
      (text_model): T5Gemma2TextEncoder(
        (embed_tokens): T5Gemma2TextScaledWordEmbedding(262144, 640, padding_idx=0)
        (norm): T5Gemma2RMSNorm((640,), eps=1e-06)
        (layers): ModuleList(
          (0-17): 18 x T5Gemma2EncoderLayer(
            (self_attn): T5Gemma2SelfAttention(
              (q_proj): Linear(in_features=640, out_features=1024, bias=False)
              (k_proj): Linear(in_features=640, out_features=256, bias=False)
              (v_proj): Linear(in_features=640, out_features=256, bias=False)
              (o_proj): Linear(in_features=1024, out_features=640, bias=False)
              (q_norm): T5Gemma2RMSNorm((256,), eps=1e-06)
              (k_norm): T5Gemma2RMSNorm((256,), eps=1e-06)
            )
            (pre_self_attn_layernorm): T5Gemma2RMSNorm((640,), eps=1e-06)
            (post_self_attn_layernorm): T5Gemma2RMSNorm((640,), eps=1e-06)
            (m

In [13]:
#original_model = prepare_model_for_kbit_training(original_model)


It is possible to pull out the number of model parameters and find out how many of them are trainable. The following function can be used to do that, at this stage, you do not need to go into details of it.

In [15]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: "
        f"{100 * trainable_model_params / all_model_params:.2f}%"
    )

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 786029296
all model parameters: 786029296
percentage of trainable model parameters: 100.00%


<a name='1.3'></a>
### 1.3 - Test the Model with Zero Shot Inferencing

Test the model with zero-shot inferencing. You can see that the model struggles to produce a good Arabic summary without task-specific training, but it does understand the input — which indicates the model can be fine-tuned to the task at hand.

In [32]:
index = 10

article                = dataset["test"][index][TEXT_COL]
human_baseline_summary = dataset["test"][index][SUMMARY_COL]

prompt = f"summarize: {article}"

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
output = tokenizer.decode(
    original_model.generate(
        **inputs,
        generation_config=GenerationConfig(max_new_tokens=128, num_beams=4),
    )[0],
    skip_special_tokens=True
)

dash_line = "-".join("" for x in range(100))
print(dash_line)
print(f"INPUT ARTICLE:\n{article[:500]}...")
print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}\n")
print(dash_line)
print(f"MODEL GENERATION - ZERO SHOT:\n{output}")

---------------------------------------------------------------------------------------------------
INPUT ARTICLE:
إذا كنت في علاقة فمن المحتمل دائما أنك ستواجه لحظات من الرغبات غير المتوافقة التي لابد أن تشهدها العلاقة. القليل من الناس هم من يكون لديهم الرغبات والاحتياجات متماشية ومتناغمة تماما، لذا سيتحتم عليكما معا إجراء المحادثات المفتوحة والصادقة حول احتياجات كل منكما.  قد تبدو مناقشة تفضيلاتك أمرا غير مريحا في البداية لكنها يمكن أن تصبح طريقة جيدة للاتحاد مع شريكك.  احصلا على الوقت المناسب للحديث عندما لا يوجد مايشتتكما أو يقطع حديثكما، يجب أن يشعر كلاكما بالتركيز على ما تحتاجه العلاقة، وذلك لا يمكن له...
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
إذا كنت في علاقة، قد تواجه رغبات غير متطابقة. اجرِ محادثات صريحة حول احتياجاتكما. استخدم جمل تعبر بصيغة المتكلم "أنا". حافظ على التواصل الرومانسي وقم بتعيين مواعيد للقاءات الجنسية. تذكر أن الإثارة الجسدية مهمة للنساء. تعاونا لتجربة أنشطة جنسية جديدة وبناء ر

<a name='2'></a>
## 2 - Perform Parameter Efficient Fine-Tuning (PEFT) with LoRA


In [33]:
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 128

def tokenize_function(example):
    inputs = tokenizer(
        [f"summarize: {t}" for t in example[TEXT_COL]],
        max_length = MAX_INPUT_LEN,
        truncation = True,
        padding    = False,
    )
    targets = tokenizer(
        text_target = example[SUMMARY_COL],
        max_length  = MAX_TARGET_LEN,
        truncation  = True,
        padding     = False,
    )
    # Replace pad token id with -100 so loss ignores padding
    inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in targets["input_ids"]
    ]
    return inputs

In [34]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched        = True,
    remove_columns = dataset["train"].column_names,
)

print(f"Train      : {len(tokenized_datasets['train']):,} samples")
print(f"Validation : {len(tokenized_datasets['validation']):,} samples")
print(f"Test       : {len(tokenized_datasets['test']):,} samples")
print(f"\nFeatures   : {list(tokenized_datasets['train'].features.keys())}")

Train      : 3,526 samples
Validation : 437 samples
Test       : 444 samples

Features   : ['input_ids', 'attention_mask', 'labels']


<a name='2.2'></a>
### 2.2 - Setup the PEFT/LoRA Model for Fine-Tuning

You need to set up the PEFT/LoRA model for fine-tuning with a new layer/parameter adapter. Using PEFT/LoRA, you are freezing the underlying LLM and only training the adapter. Have a look at the LoRA configuration below. Note the rank (`r`) hyperparameter, which defines the rank/dimension of the adapter to be trained.

In [35]:
# Run this to print all layer names in the model
for name, module in original_model.named_modules():
    print(name)


model
model.encoder
model.encoder.text_model
model.encoder.text_model.embed_tokens
model.encoder.text_model.norm
model.encoder.text_model.layers
model.encoder.text_model.layers.0
model.encoder.text_model.layers.0.self_attn
model.encoder.text_model.layers.0.self_attn.q_proj
model.encoder.text_model.layers.0.self_attn.k_proj
model.encoder.text_model.layers.0.self_attn.v_proj
model.encoder.text_model.layers.0.self_attn.o_proj
model.encoder.text_model.layers.0.self_attn.q_norm
model.encoder.text_model.layers.0.self_attn.k_norm
model.encoder.text_model.layers.0.pre_self_attn_layernorm
model.encoder.text_model.layers.0.post_self_attn_layernorm
model.encoder.text_model.layers.0.mlp
model.encoder.text_model.layers.0.mlp.gate_proj
model.encoder.text_model.layers.0.mlp.up_proj
model.encoder.text_model.layers.0.mlp.down_proj
model.encoder.text_model.layers.0.mlp.act_fn
model.encoder.text_model.layers.0.mlp.dropout
model.encoder.text_model.layers.0.pre_feedforward_layernorm
model.encoder.text_mod

In [36]:
lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.SEQ_2_SEQ_LM
)

Add LoRA adapter layers/parameters to the original LLM to be trained.

In [37]:
peft_model = get_peft_model(original_model, lora_config)
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 5935104
all model parameters: 791964400
percentage of trainable model parameters: 0.75%


<a name='2.3'></a>
### 2.3 - Train PEFT Adapter

Define training arguments and create a `Seq2SeqTrainer` instance.

In [38]:
OUTPUT_DIR = f"./t5gemma-arabic-peft-{str(int(time.time()))}"

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model              = None,
    padding            = True,
    pad_to_multiple_of = 8,
    label_pad_token_id = -100,
)

peft_training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 1e-3,
    fp16                        = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET_LEN,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    warmup_steps                = 50,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    logging_steps               = 10,
    dataloader_num_workers      = 2,

)

peft_trainer = Seq2SeqTrainer(
    model            = peft_model,
    args             = peft_training_args,
    train_dataset    = tokenized_datasets["train"],
    eval_dataset     = tokenized_datasets["validation"],
    processing_class = tokenizer,
    data_collator    = data_collator,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)



Now everything is ready to train the PEFT adapter and save the model.

In [39]:
peft_trainer.train()


Epoch,Training Loss,Validation Loss
1,2.003264,2.047860
2,1.760103,1.919587
3,1.555031,1.907799


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=663, training_loss=1.7655406156456488, metrics={'train_runtime': 3051.0763, 'train_samples_per_second': 3.467, 'train_steps_per_second': 0.217, 'total_flos': 2.011911646606848e+16, 'train_loss': 1.7655406156456488, 'epoch': 3.0})

In [51]:
repo_name = "t5gemma-arabic-summarization-peft"
peft_trainer.model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/ZeyadAhmedMostafa/t5gemma-arabic-summarization-peft/commit/9a2e26a3523389bbf3e02977c09131c7bc636194', commit_message='Upload tokenizer', commit_description='', oid='9a2e26a3523389bbf3e02977c09131c7bc636194', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ZeyadAhmedMostafa/t5gemma-arabic-summarization-peft', endpoint='https://huggingface.co', repo_type='model', repo_id='ZeyadAhmedMostafa/t5gemma-arabic-summarization-peft'), pr_revision=None, pr_num=None)

In [41]:

peft_model_path = "./t5gemma-arabic-peft-checkpoint"
peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

print(f"\nPEFT adapter saved to: {peft_model_path}")


PEFT adapter saved to: ./t5gemma-arabic-peft-checkpoint


In [42]:
import gc
torch.cuda.empty_cache()
gc.collect()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Free VRAM: 10.10 GB


Load the saved PEFT model back for inference.

In [43]:
# Free training memory before loading inference model
#del peft_trainer
#gc.collect()
#torch.cuda.empty_cache()

peft_model_loaded = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype = torch.bfloat16,
    device_map  = "auto",
)
peft_model_loaded = PeftModel.from_pretrained(
    peft_model_loaded,
    peft_model_path,
    torch_dtype = torch.bfloat16,
)
peft_model_loaded.eval()
print("PEFT model loaded for inference.")

Loading weights:   0%|          | 0/911 [00:00<?, ?it/s]

PEFT model loaded for inference.


<a name='2.4'></a>
### 2.4 - Evaluate the Model Qualitatively (Human Evaluation)

As with many GenAI applications, a qualitative approach where you ask yourself the question *"Is my model behaving the way it is supposed to?"* is usually a good starting point. In the example below (the same one we started this notebook with), you can see how the fine-tuned model is able to create a reasonable Arabic summary compared to the original model's zero-shot attempt.

In [53]:
index = 8

article                = dataset["test"][index][TEXT_COL]
human_baseline_summary = dataset["test"][index][SUMMARY_COL]

prompt    = f"summarize: {article}"
input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)

original_model_outputs = original_model.generate(
    input_ids=input_ids,
    generation_config=GenerationConfig(max_new_tokens=128, num_beams=4)
)
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

peft_model_outputs = peft_model_loaded.generate(
    input_ids=input_ids,
    generation_config=GenerationConfig(max_new_tokens=128, num_beams=4)
)
peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

dash_line = "-".join("" for x in range(100))
print(dash_line)
print(f"INPUT ARTICLE:\n{article[:500]}...")
print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL:\n{original_model_text_output}")
print(dash_line)
print(f"PEFT MODEL:\n{peft_model_text_output}")

---------------------------------------------------------------------------------------------------
INPUT ARTICLE:
. في حال كنت تحاول أن تحب نفسك بصورة أكبر، فقد تواجه شعور بالخزي والعار وإحساس بأنك أقل من أن تحب ذاتك.حاول الكتابة عن مشاعرك العميقة التي تحاول إخفائها والتي تكون سببا أساسيا في شعورك بالخزي. عندما تقوم بالتدوين عن مشاعرك وعن الصعوبات والآلام التي تواجهها، ستشعر بشكل أفضل تجاه ذاتك.  حاول إعادة تكوين التجربة المريرة لحظة بلحظة من منظور مختلف للموقف. فكر فيما تعلمت من الموقف، وكن رفيقا بالحكم على تصرفك تجاهه. وتذكر أن هناك العديد من الأساليب الأخرى للرد على الموقف. حاول أن تقوم بالتدوين عن تصرفا...
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
في حال كنت تحاول حب نفسك، قد تواجه شعورا بالخزي. حاول الكتابة عن مشاعرك العميقة وتدوين تجاربك. إعادة تشكيل التجارب من منظور مختلف والتعلم منها يمكن أن يساعدك. تقبل ذاتك واحترفها، واستشرف المستقبل، ولا تنسى تطوير نمط حياة صحي. استخدم أدوات الاسترخاء مثل التأ

Test a few more examples from the test set to get a broader picture of the improvement.

In [45]:
dash_line = "-".join("" for x in range(100))

for index in [0, 1, 2]:
    article                = dataset["test"][index][TEXT_COL]
    human_baseline_summary = dataset["test"][index][SUMMARY_COL]

    prompt    = f"summarize: {article}"
    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)

    original_outputs = tokenizer.decode(
        original_model.generate(
            input_ids=input_ids,
            generation_config=GenerationConfig(max_new_tokens=128, num_beams=4)
        )[0],
        skip_special_tokens=True
    )
    peft_outputs = tokenizer.decode(
        peft_model_loaded.generate(
            input_ids=input_ids,
            generation_config=GenerationConfig(max_new_tokens=128, num_beams=4)
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print(f"Example #{index + 1}")
    print(dash_line)
    print(f"ARTICLE (first 400 chars):\n{article[:400]}...")
    print(dash_line)
    print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}")
    print(dash_line)
    print(f"ORIGINAL MODEL:\n{original_outputs}")
    print(dash_line)
    print(f"PEFT MODEL:\n{peft_outputs}")
    print()

---------------------------------------------------------------------------------------------------
Example #1
---------------------------------------------------------------------------------------------------
ARTICLE (first 400 chars):
في العام 1061 ميلادية كانت صقلية لا تزال جزيرة شبه عربية، لكنها كانت مجزأة إلى
خمس إمارات منقسمة على نفسها، وإضافة لذلك تسبب التنازع بين الأمراء، والتنافس
بين العرب والأمازيغ في الجزيرة الواقعة جنوبي إيطاليا، في تمكّن ملك النورمان
روجر الأول من إخضاع أغلب الجزيرة لحكمه، وبحلول العام 1072 أصبحت العاصمة
الصقلية باليرمو تحت سيطرة الرومان سيطرة كاملة. لكن أفول نجم العرب السياسي عن
صقلية لم يغيّب الثقا...
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
في 1061، كانت صقلية مجزأة إلى خمس إمارات مع تنافس عربي وأمازيغي. استطاع الملك النورماني روeger الأول السيطرة عليها، وأصبحت باليرمو عاصمتها عام 1072. رغم فقدان العرب للسلطة السياسية، ظلوا الثقافة الرئيسية، وازدهرت فيها الأدب والعلم حتى

<a name='2.5'></a>
### 2.5 - Evaluate the Model Quantitatively (with ROUGE Metric)

The [ROUGE metric](https://en.wikipedia.org/wiki/ROUGE_(metric)) (**R**ecall-**O**riented **U**nderstudy for **G**isting **E**valuation) measures the overlap between generated and reference summaries. We evaluate on the test set and compute ROUGE-1, ROUGE-2, and ROUGE-L for both the original and PEFT models.

In [46]:
rouge = evaluate.load("rouge")

# Use a subset for speed — increase N_SAMPLES up to 444 for full test-set evaluation
N_SAMPLES = 50

human_baseline_summaries = []
original_model_summaries = []
peft_model_summaries     = []

print(f"Generating summaries for {N_SAMPLES} test samples ...")
for idx in tqdm(range(N_SAMPLES)):
    article = dataset["test"][idx][TEXT_COL]
    ref     = dataset["test"][idx][SUMMARY_COL]

    prompt    = f"summarize: {article}"
    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)

    orig_out = tokenizer.decode(
        original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=128))[0],
        skip_special_tokens=True
    )
    peft_out = tokenizer.decode(
        peft_model_loaded.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=128))[0],
        skip_special_tokens=True
    )

    human_baseline_summaries.append(ref)
    original_model_summaries.append(orig_out)
    peft_model_summaries.append(peft_out)

print("Done.")

Generating summaries for 50 test samples ...


100%|██████████| 50/50 [14:40<00:00, 17.62s/it]

Done.


In [47]:
zipped_summaries = list(zip(human_baseline_summaries, original_model_summaries, peft_model_summaries))

df = pd.DataFrame(
    zipped_summaries,
    columns=["human_baseline_summaries", "original_model_summaries", "peft_model_summaries"]
)
df

,human_baseline_summaries,original_model_summaries,peft_model_summaries
0,في 1061، كانت صقلية مجزأة إلى خمس إمارات مع تن...,في عهد ملوك النورمان، لم تكن جزيرة شبه عربية، ...,فى 1061 ميلادية، كانت صقلية شبه عربية. خلال حك...
1,قررت الشركة القطرية لتموين الطائرات الحكومية إ...,استمرت الاتفاقية على توزيع وجبات الطعام والمشر...,اتفقت الشركة القطرية لتموين الطائرات مع مركز ح...
2,دعا محمد علي لتنظيم مظاهرات في ذكرى ثورة 25 ين...,دعوت محمد علي لمحمد علي للدعوة للنزول في ذكرى ...,دعوة محمد علي لنزول في ذكرى الثورة أدى إلى نشط...
3,إذا أردت للفتاة أن تفتقدك في غيابك، يجب تقليل ...,إذا أردت للفتاة أن تفتقدك في غيابك، يجب أن تخر...,في غيابك، حاول عدم تقف عندها لأسبوعين، واستخدم...
4,فيما يتعلق بتخطيطك للاستثمار في الأسهم، يجب تح...,أهداف الاستثمار يجب توثيقها بالدولار وجمعها من...,هناك عدة طرق للتحكم في الاستثمارات، منها تبسيط...
5,([Ar])4s² 3d¹⁰,تسمى الذرات على طول الحافة اليمنى من الجدول ال...,تسمى الذرات على طول الحافة اليمنى من الجدول ال...
6,فيما يتعلق بالنظم الغذائية النباتية:\n\n1. ضما...,تتضمن النصائح تقديم النظام الغذائي النباتي، تن...,يحتوي النص على إرشادات حول كيفية تقديم الأطعمة...
7,أثبتت حالة وفاة رضيع أمريكي بسبب كورونا وجود ز...,أثارت وفاة رضيع أميركي كورونا انتهاكاً للمصروف...,أكد الدكتور أسامة أبو الرب إصابة رضيع أميركي ب...
8,في حال كنت تحاول حب نفسك، قد تواجه شعورا بالخز...,تتواجه الحاجة العميقة للمواقف الحياتية، وقد يك...,في حال تريد أن تحب نفسك، فقد تواجه عاراً للإحس...
9,تتضمن وجبات الرضع التقليدية فواكه مثل الموز، ا...,تحضير وجبات الرضع التقليدية تحتاج إلى سهولة ال...,تتوفر وجبات الرضع التقليدية من الفواكه والخضرو...


Compute ROUGE score for this subset of the data.

In [48]:
original_model_results = rouge.compute(
    predictions    = original_model_summaries,
    references     = human_baseline_summaries,
    use_aggregator = True,
    use_stemmer    = False,
)

peft_model_results = rouge.compute(
    predictions    = peft_model_summaries,
    references     = human_baseline_summaries,
    use_aggregator = True,
    use_stemmer    = False,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("PEFT MODEL:")
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.04), 'rouge2': np.float64(0.03), 'rougeL': np.float64(0.04), 'rougeLsum': np.float64(0.04)}
PEFT MODEL:
{'rouge1': np.float64(0.07298701298701299), 'rouge2': np.float64(0.04158730158730158), 'rougeL': np.float64(0.07286255411255413), 'rougeLsum': np.float64(0.07311147186147188)}


In [56]:
# Extract the average scores from your previous results
original_rouge = {
    'rouge1': original_model_results['rouge1'], 
    'rouge2': original_model_results['rouge2'], 
    'rougeL': original_model_results['rougeL'], 
    'rougeLsum': original_model_results['rougeLsum']
}

peft_rouge = {
    'rouge1': peft_model_results['rouge1'], 
    'rouge2': peft_model_results['rouge2'], 
    'rougeL': peft_model_results['rougeL'], 
    'rougeLsum': peft_model_results['rougeLsum']
}

print("="*80)
print(f"{'Metric':<12} | {'Original':<10} | {'PEFT':<10} | {'Absolute Imp.':<15} | {'Percentage Imp. (%)'}")
print("-" * 80)

for key in original_rouge:
    orig = float(original_rouge[key])
    peft = float(peft_rouge[key])
    
    # Calculate Absolute and Percentage Improvement
    abs_diff = peft - orig
    pct_imp = (abs_diff / orig) * 100
    
    print(f"{key:<12} | {orig:<10.4f} | {peft:<10.4f} | +{abs_diff:<14.4f} | +{pct_imp:.2f}%")
print("="*80)

Metric       | Original   | PEFT       | Absolute Imp.   | Percentage Imp. (%)
--------------------------------------------------------------------------------
rouge1       | 0.0400     | 0.0730     | +0.0330         | +82.47%
rouge2       | 0.0300     | 0.0416     | +0.0116         | +38.62%
rougeL       | 0.0400     | 0.0729     | +0.0329         | +82.16%
rougeLsum    | 0.0400     | 0.0731     | +0.0331         | +82.78%


In [62]:
bertscore = evaluate.load("bertscore")

print("Evaluating Original Model with BERTScore...")
original_bert_results = bertscore.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
    lang="ar",
    model_type="aubmindlab/bert-base-arabertv02",
    num_layers=9,          
)

print("Evaluating PEFT Model with BERTScore...")
peft_bert_results = bertscore.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries,
    lang="ar",
    model_type="aubmindlab/bert-base-arabertv02",
    num_layers=9,          
)

# Extract and compute the mean F1 scores
orig_f1_mean = np.mean(original_bert_results['f1'])
peft_f1_mean = np.mean(peft_bert_results['f1'])

print("\n" + "="*50)
print("BERTScore F1 Results (Semantic Similarity)")
print("="*50)
print(f"Original Model F1 : {orig_f1_mean:.4f}")
print(f"PEFT Model F1     : {peft_f1_mean:.4f}")

bert_abs_diff = peft_f1_mean - orig_f1_mean
bert_pct_imp  = (bert_abs_diff / orig_f1_mean) * 100
print(f"\nImprovement       : +{bert_abs_diff:.4f} absolute (+{bert_pct_imp:.2f}%)")
print("="*50)

Evaluating Original Model with BERTScore...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating PEFT Model with BERTScore...

BERTScore F1 Results (Semantic Similarity)
Original Model F1 : 0.6138
PEFT Model F1     : 0.6108

Improvement       : +-0.0031 absolute (+-0.50%)
